# Valence evidence by operation

Interactive, cell-by-cell version of `_interactive_notebooks/plot_valence_evidence_by_operation.py`
-- built on the reusable functions in `_interactive_notebooks/decoding_results_toolkit.py`.
Every filter, condition, stat window, and stat method is a plain variable in
the **Configuration** cell below; re-run from there down after changing any
of them.

Reads each subject's own `decoding_results.csv` (raw, one row per decoded
TR) and, for each operation category (`regressor_label` -- e.g.
maintain/suppress/switch/clear), shows the classifier's *self*-evidence
(`evidence_<that row's own true category>`) over time, split into WMpos
("pos") vs. WMneg ("neg") trials -- per stimulus (face/place, a sanity
check) and collapsed across stimulus (the average of the two stimulus
means, the panel that actually matters for the pos-vs-neg question). A
maintain-baseline-subtracted view and a binned, one-tailed paired
significance test (pos > neg by default) are built the same way, with
significant windows marked directly on the timecourse plots.

In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import pandas as pd

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))  # run from _interactive_notebooks/, repo root one level up
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from workflows.generate_report import resolve_desc, list_subject_dirs, parse_subjects_arg
from _interactive_notebooks.decoding_results_toolkit import (
    load_decoding_results, select_evidence_value, derive_label,
    aggregate_by_subject_window, average_across_groups, subtract_baseline,
    bin_by_size, bin_by_edges, compare_conditions_by_bin, plot_conditions,
    annotate_significance, STAT_METHODS,
)

%matplotlib inline

## Configuration -- edit these

Everything below is a plain variable, not a CLI flag -- change any of them
and re-run from here down.

In [ ]:
# -- where the data is --
ANALYSIS_OUTPUT_DIR = "/path/to/analysis/output"   # same value passed to mvpa_workflow.py
DESC = "gm_operation_kfold_classifier-2"           # classifier folder name (model.desc, sanitized)
CONFIG_PATH = None                                 # optional: a config JSON path instead of DESC
                                                    # (its model.desc is read via resolve_desc, so this
                                                    # can never drift out of sync with what the workflow wrote)

# subjects to include -- None for every subject found, or a comma-separated
# string ("001,004,010"), or a path to a text file listing them (one per
# line and/or comma-separated) -- same rules as generate_report.py --subjects
SUBJECTS = None

# -- filters / condition labels (see _interactive_notebooks/decoding_results_toolkit.py's
# apply_filters/derive_label for the full range of criteria types: exact
# value, list, regex, or a callable predicate) --
VALENCE_MAPPING = {"pos": "WMpos", "neg": "WMneg"}      # task -> condition label (exact match)
STIMULUS_MAPPING = {"face": "face", "place": "place"}   # trial_type -> group label (regex, case-insensitive)
VALENCE_COLORS = {"pos": "red", "neg": "blue"}          # plotting colors, same keys as VALENCE_MAPPING
BASELINE_OPERATION = "maintain"                         # subtracted from every other operation below

# per-operation colors, matching this project's established color coding
# elsewhere (report overlay configs, prior figures) -- used by the "full
# timetrace by operation" section below. No standard matplotlib/seaborn
# qualitative palette happens to put these 4 hues in this exact order, so
# this is just an explicit literal mapping rather than a palette lookup.
OPERATION_COLORS = {"maintain": "green", "switch": "blue", "suppress": "red", "clear": "orange"}

# -- stat windows -- set BIN_EDGES to use explicit (tr_start, tr_end) pairs
# instead of uniform-size bins (BIN_EDGES overrides BIN_SIZE when not None) --
BIN_SIZE = 3
BIN_EDGES = None  # e.g. [(1, 3), (4, 6), (7, 9), (10, 16)]

# -- stat method: any key in STAT_METHODS ("ttest_1samp_diff", "ttest_rel",
# "wilcoxon"), or your own callable(a_vals, b_vals, alternative) -> dict --
STAT_METHOD = "ttest_1samp_diff"
ALTERNATIVE = "greater"  # "greater" tests pos > neg; "less"/"two-sided" also available
SIG_ALPHA = 0.05         # significance threshold for the bar/star annotations below

# -- where to save the PDF + stats CSV (last cell only -- everything above
# it is display-only, nothing is written to disk until you run that cell) --
OUTPUT_DIR = "./valence_evidence_analysis" 

## Load data

In [ ]:
desc = resolve_desc(None, CONFIG_PATH) if CONFIG_PATH else DESC
subjects_arg = parse_subjects_arg(SUBJECTS) if SUBJECTS else None
subjects = list_subject_dirs(ANALYSIS_OUTPUT_DIR, desc, subjects=subjects_arg)
print(f"Scope: {len(subjects)} subject(s): {subjects}")

raw = load_decoding_results(ANALYSIS_OUTPUT_DIR, desc, subjects)
df = select_evidence_value(raw, value="self")
df = derive_label(df, "task", VALENCE_MAPPING, new_col="valence")
df = derive_label(df, "trial_type", STIMULUS_MAPPING, new_col="stimulus", regex=True)

operations = sorted(df["regressor_label"].unique(), key=lambda o: (o != BASELINE_OPERATION, o))
print("Operations found:", operations)
df.head()

## Full timetrace: classifier evidence by operation

One line per operation (each operation's own self-evidence, collapsed
across valence and stimulus -- the standard decoding-accuracy-style
timecourse), colored per `OPERATION_COLORS` above -- a quick overview
before the pos/neg valence breakdown below.

In [ ]:
operation_wide = aggregate_by_subject_window(df, condition_col="regressor_label")

fig, ax = plt.subplots(figsize=(7, 4.5))
plot_conditions(ax, operation_wide, OPERATION_COLORS, title=None, ylabel="classifier evidence (own true category)")
ax.set_title(f"{desc}: classifier evidence by operation", fontsize=12, fontweight="bold")
plt.show()

## Aggregate: per-subject, per-TR means by condition

One `{"face": ..., "place": ..., "collapsed": ...}` dict per operation --
`"collapsed"` is the equal-weighted average of the face-only and place-only
means (see `average_across_groups`), not a direct pool of every trial.

In [ ]:
def per_operation_means(operation):
    op_df = df[df["regressor_label"] == operation]
    per_stimulus = {
        stim: aggregate_by_subject_window(op_df[op_df["stimulus"] == stim], condition_col="valence")
        for stim in STIMULUS_MAPPING
    }
    return {**per_stimulus, "collapsed": average_across_groups(per_stimulus)}


means_by_operation = {op: per_operation_means(op) for op in operations}
list(means_by_operation)

## Raw pos vs. neg evidence, per operation

Re-run this cell after changing `VALENCE_MAPPING`/`STIMULUS_MAPPING`/
`VALENCE_COLORS` above.

In [ ]:
def render_operation_figure(op_means, suptitle):
    fig, axes = plt.subplots(1, 3, figsize=(12, 4), sharex=True, sharey=True)
    for ax, key in zip(axes, ("face", "place", "collapsed")):
        plot_conditions(ax, op_means[key], VALENCE_COLORS, title=key)
    axes[0].set_ylabel("classifier evidence (own true category)")
    fig.suptitle(suptitle, fontsize=13, fontweight="bold")
    fig.tight_layout(rect=[0, 0, 1, 0.93])
    return fig


for op in operations:
    render_operation_figure(means_by_operation[op], f"{op}: pos vs. neg classifier evidence")
    plt.show()

## Maintain-baseline-subtracted views

`suppress`/`switch`/`clear` with `BASELINE_OPERATION`'s own (subject- and
TR-matched) evidence subtracted out first -- does a removal operation's
pos/neg pattern look different from the baseline's own, once its pattern is
removed?

In [ ]:
adjusted_by_operation = {}
if BASELINE_OPERATION in means_by_operation:
    baseline = means_by_operation[BASELINE_OPERATION]
    for op in operations:
        if op == BASELINE_OPERATION:
            continue
        adjusted = {
            scope: subtract_baseline(means_by_operation[op][scope], baseline[scope])
            for scope in ("face", "place", "collapsed")
        }
        adjusted_by_operation[op] = adjusted
        render_operation_figure(adjusted, f"{op} minus {BASELINE_OPERATION} baseline: pos vs. neg classifier evidence")
        plt.show()
else:
    print(f"(!) no {BASELINE_OPERATION!r} operation found -- skipping baseline-subtracted views")

## Significance testing (binned, adjustable)

Re-run after changing `BIN_SIZE`/`BIN_EDGES`/`STAT_METHOD`/`ALTERNATIVE`
above. Each row is one (operation, stimulus scope, bin) -- `mean_diff` is
this bin's mean (pos - neg) difference score across subjects, `p_value` its
significance under `STAT_METHOD`.

In [ ]:
def bin_wide_df(wide_df):
    if BIN_EDGES is not None:
        return bin_by_edges(wide_df, edges=BIN_EDGES)
    return bin_by_size(wide_df, bin_size=BIN_SIZE)


def stats_for(means_dict, extra_cols):
    rows = []
    for op, op_means in means_dict.items():
        for scope in ("face", "place", "collapsed"):
            per_subject_bin, bins_meta = bin_wide_df(op_means[scope])
            bin_stats = compare_conditions_by_bin(
                per_subject_bin, "pos", "neg", method=STAT_METHOD, alternative=ALTERNATIVE, bins_meta=bins_meta,
            )
            if bin_stats.empty:
                continue
            bin_stats.insert(0, "operation", op)
            bin_stats.insert(1, "stimulus_scope", scope)
            for col, val in extra_cols.items():
                bin_stats[col] = val
            rows.append(bin_stats)
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()


raw_stats = stats_for(means_by_operation, {"baseline_subtracted": False})
raw_stats.style.apply(lambda col: ["background-color: #ffff99" if v < 0.05 else "" for v in col], subset=["p_value"])

In [ ]:
baseline_stats = stats_for(adjusted_by_operation, {"baseline_subtracted": True})
baseline_stats.style.apply(lambda col: ["background-color: #ffff99" if v < 0.05 else "" for v in col], subset=["p_value"]) if not baseline_stats.empty else baseline_stats

### Significance visualization

The collapsed panel (pos vs. neg) for each operation, with a bar + `*`
marked above every bin `SIG_ALPHA`-significant in the table above -- ties
the stats directly to the timecourse rather than reading them separately.

In [ ]:
def render_collapsed_with_significance(wide_df, bin_stats_for_panel, title):
    fig, ax = plt.subplots(figsize=(6, 4.5))
    plot_conditions(ax, wide_df, VALENCE_COLORS, title=None, ylabel="classifier evidence (own true category)")
    annotate_significance(ax, bin_stats_for_panel, alpha=SIG_ALPHA)
    ax.set_title(title, fontsize=11, fontweight="bold")
    return fig


for op in operations:
    op_stats = raw_stats[(raw_stats["operation"] == op) & (raw_stats["stimulus_scope"] == "collapsed")]
    render_collapsed_with_significance(means_by_operation[op]["collapsed"], op_stats, f"{op}: pos vs. neg (collapsed)")
    plt.show()

In [ ]:
for op, adjusted in adjusted_by_operation.items():
    op_stats = baseline_stats[(baseline_stats["operation"] == op) & (baseline_stats["stimulus_scope"] == "collapsed")]
    render_collapsed_with_significance(
        adjusted["collapsed"], op_stats, f"{op} minus {BASELINE_OPERATION}: pos vs. neg (collapsed)"
    )
    plt.show()

## Save outputs (optional)

Writes the same PDF + stats CSV `plot_valence_evidence_by_operation.py`'s
CLI produces, to `OUTPUT_DIR` (set above) -- run this once you're happy
with the configuration.

In [ ]:
from matplotlib.backends.backend_pdf import PdfPages

os.makedirs(OUTPUT_DIR, exist_ok=True)
pdf_path = os.path.join(OUTPUT_DIR, f"{desc}_valence_evidence.pdf")

with PdfPages(pdf_path) as pdf:
    for op in operations:
        fig = render_operation_figure(means_by_operation[op], f"{op}: pos vs. neg classifier evidence")
        pdf.savefig(fig)
        plt.close(fig)
    for op, adjusted in adjusted_by_operation.items():
        fig = render_operation_figure(adjusted, f"{op} minus {BASELINE_OPERATION} baseline: pos vs. neg classifier evidence")
        pdf.savefig(fig)
        plt.close(fig)
print(f"PDF written to: {pdf_path}")

stats_df = pd.concat([raw_stats, baseline_stats], ignore_index=True) if not baseline_stats.empty else raw_stats
stats_path = os.path.join(OUTPUT_DIR, f"{desc}_valence_evidence_stats.csv")
stats_df.to_csv(stats_path, index=False)
print(f"Stats table written to: {stats_path}")